# Diabetic Retinopathy Severity Grading — APTOS 2019

**Transfer learning with EfficientNetB0 for 5-class ordinal grading of retinal fundus photographs.**

This notebook trains and evaluates a model that grades diabetic retinopathy (DR) severity into the
five clinical classes used by the [APTOS 2019 Blindness Detection](https://www.kaggle.com/competitions/aptos2019-blindness-detection)
competition:

| Grade | Class | Meaning |
|:-----:|:------|:--------|
| 0 | `No_DR` | No diabetic retinopathy |
| 1 | `Mild` | Microaneurysms only |
| 2 | `Moderate` | More extensive microaneurysms / haemorrhages |
| 3 | `Severe` | Extensive haemorrhages, venous beading |
| 4 | `Proliferative_DR` | Neovascularisation / vitreous haemorrhage |

**It runs top to bottom with no manual steps besides (optionally) providing a Kaggle API key.**
If no key is supplied, the notebook automatically falls back to a public Hugging Face mirror of the
dataset, so it always runs.

> ⚠️ **Not a medical device.** Research/education only — not for clinical use.

---
**➡️ Before running:** enable the GPU via **Runtime → Change runtime type → Hardware accelerator → GPU**.

## Why Quadratic Weighted Kappa (QWK)?

DR severity is **ordinal** — the grades have a natural order, and mistakes are not equally bad.
Confusing *Severe* (3) with *Proliferative* (4) is a minor slip; confusing *Proliferative* (4) with
*No_DR* (0) could send a patient who needs urgent treatment home untreated.

Plain **accuracy** treats every misclassification identically. **Quadratic Weighted Kappa** instead
penalises each error by the **squared distance** between the predicted and true grade, and corrects
for chance agreement. It is the official APTOS 2019 metric, so we report it as our headline number
alongside accuracy, per-class precision/recall, ROC-AUC and a confusion matrix.

In [ ]:
# --- Dependencies (TensorFlow, scikit-learn, matplotlib are pre-installed on Colab) ---
!pip -q install kagglehub "huggingface_hub>=0.23" "opencv-python-headless>=4.8,<5.0" 2>/dev/null
print("Dependencies ready.")

In [ ]:
import os, json, warnings
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, cohen_kappa_score,
                             roc_auc_score, roc_curve)
from sklearn.preprocessing import label_binarize

warnings.filterwarnings("ignore")
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices('GPU'))

# ------------------------- Configuration -------------------------
SEED          = 42
IMG_SIZE      = 224
BATCH_SIZE    = 32
VAL_SPLIT     = 0.15
TEST_SPLIT    = 0.15
DROPOUT       = 0.4
CIRCLE_CROP   = True

HEAD_EPOCHS       = 25
HEAD_LR           = 1e-3
HEAD_PATIENCE     = 6
FINETUNE_EPOCHS   = 20
FINETUNE_LR       = 1e-4
FINETUNE_UNFREEZE = 40
FINETUNE_PATIENCE = 6

CLASS_NAMES = ["No_DR", "Mild", "Moderate", "Severe", "Proliferative_DR"]
NUM_CLASSES = len(CLASS_NAMES)
tf.keras.utils.set_random_seed(SEED)

## 1. Get the data

**Primary source — Kaggle.** Upload your `kaggle.json` API token (Kaggle → *Account* → *Create New
API Token*) when prompted. **Fallback — Hugging Face mirror** (no credentials needed), used
automatically if a Kaggle key isn't available so the notebook always runs.

In [ ]:
def setup_kaggle_credentials():
    """Locate a kaggle.json token (Colab upload / Drive / env) and export it."""
    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        return True
    for cand in ["/content/kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"),
                 "/root/.kaggle/kaggle.json"]:
        if os.path.exists(cand):
            os.environ["KAGGLE_CONFIG_DIR"] = str(Path(cand).parent)
            return True
    try:
        from google.colab import files
        print("Upload your kaggle.json (or press Cancel to use the public mirror):")
        up = files.upload()
        if "kaggle.json" in up:
            os.makedirs("/root/.kaggle", exist_ok=True)
            with open("/root/.kaggle/kaggle.json", "wb") as f:
                f.write(up["kaggle.json"])
            os.chmod("/root/.kaggle/kaggle.json", 0o600)
            os.environ["KAGGLE_CONFIG_DIR"] = "/root/.kaggle"
            return True
    except Exception:
        pass
    return False


def download_dataset():
    """Return (train_csv, train_images_dir); Kaggle first, HF mirror fallback."""
    if setup_kaggle_credentials():
        try:
            import kagglehub
            print("Downloading APTOS 2019 from Kaggle via kagglehub…")
            root = Path(kagglehub.competition_download("aptos2019-blindness-detection"))
            return next(root.rglob("train.csv")), next(p for p in root.rglob("train_images") if p.is_dir())
        except Exception as e:
            print("Kaggle download failed:", e, "\n→ falling back to Hugging Face mirror.")
    from huggingface_hub import snapshot_download
    print("Downloading APTOS 2019 from the public Hugging Face mirror…")
    root = Path(snapshot_download(repo_id="Tejaswini628/aptos-fundus-images",
                                  repo_type="dataset", local_dir="data/raw",
                                  allow_patterns=["data/train.csv", "data/train_images/*.png"],
                                  max_workers=16))
    return next(root.rglob("train.csv")), next(p for p in root.rglob("train_images") if p.is_dir())


CSV_PATH, IMG_DIR = download_dataset()
labels_df = pd.read_csv(CSV_PATH)
print(f"\n{len(labels_df)} labelled training images")
display(labels_df["diagnosis"].value_counts().sort_index()
        .rename(index=dict(enumerate(CLASS_NAMES))).to_frame("count"))

### Class imbalance

Roughly **half** the images are `No_DR`, while `Severe` accounts for ~5%. Left unaddressed, a model
can score high accuracy by simply under-calling severe disease. We counter this with
**inverse-frequency class weights** during training and by optimising/checkpointing on **QWK**.

In [ ]:
ax = (labels_df["diagnosis"].value_counts().sort_index()
      .rename(index=dict(enumerate(CLASS_NAMES)))
      .plot.bar(color="#3b7dd8", figsize=(7,4)))
ax.set_title("APTOS 2019 — class distribution"); ax.set_ylabel("images"); ax.set_xlabel("")
plt.tight_layout(); plt.show()

## 2. Fundus preprocessing

Fundus photos are a bright retinal disc on a black background with wide, uninformative borders.
We (1) **crop to the retina**, (2) **resize** to 224×224, and (3) apply a gentle **circular mask** to
drop bright corner artefacts. No intensity normalisation is done here — EfficientNet normalises
internally and expects `[0, 255]` inputs. We deliberately avoid heavy filtering that would obscure
the microaneurysms, haemorrhages and exudates the model (and Grad-CAM) should key on.

In [ ]:
def crop_to_retina(img, tol=7):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    mask = gray > tol
    if not mask.any():
        return img
    rows, cols = np.where(mask.any(1))[0], np.where(mask.any(0))[0]
    return img[rows.min():rows.max()+1, cols.min():cols.max()+1]

def apply_circle_mask(img):
    h, w = img.shape[:2]
    m = np.zeros((h, w), np.uint8)
    cv2.circle(m, (w//2, h//2), min(h, w)//2, 255, -1)
    return cv2.bitwise_and(img, img, mask=m)

def preprocess_fundus(img, size=IMG_SIZE, circle_crop=CIRCLE_CROP):
    img = crop_to_retina(img)
    img = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
    return apply_circle_mask(img) if circle_crop else img

# Visualise the preprocessing on one example
_ex = cv2.cvtColor(cv2.imread(str(Path(IMG_DIR)/f"{labels_df.id_code[0]}.png")), cv2.COLOR_BGR2RGB)
fig, ax = plt.subplots(1, 2, figsize=(9, 4.5))
ax[0].imshow(_ex); ax[0].set_title(f"raw {_ex.shape[1]}x{_ex.shape[0]}")
ax[1].imshow(preprocess_fundus(_ex)); ax[1].set_title(f"preprocessed {IMG_SIZE}x{IMG_SIZE}")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Preprocess every image once into a resized cache (threads — cv2 releases the GIL)
CACHE_DIR = Path("data/processed") / f"images_{IMG_SIZE}"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def _process(id_code):
    dst = CACHE_DIR / f"{id_code}.png"
    if dst.exists():
        return
    bgr = cv2.imread(str(Path(IMG_DIR) / f"{id_code}.png"))
    if bgr is None:
        return
    out = preprocess_fundus(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    cv2.imwrite(str(dst), cv2.cvtColor(out, cv2.COLOR_RGB2BGR))

todo = [i for i in labels_df.id_code if not (CACHE_DIR / f"{i}.png").exists()]
print(f"Preprocessing {len(todo)} images…")
with ThreadPoolExecutor(max_workers=os.cpu_count()) as ex:
    list(ex.map(_process, todo))

labels_df["path"] = [str(CACHE_DIR / f"{i}.png") for i in labels_df.id_code]
labels_df = labels_df[labels_df.path.map(os.path.exists)].reset_index(drop=True)
print("Cache ready:", len(labels_df), "images")

## 3. Stratified train / validation / test split

70 / 15 / 15, stratified by grade. The **test set is held out** from all training and model selection,
so the final numbers reflect genuine generalisation.

In [ ]:
train_df, temp_df = train_test_split(labels_df, test_size=VAL_SPLIT+TEST_SPLIT,
                                     stratify=labels_df.diagnosis, random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=TEST_SPLIT/(VAL_SPLIT+TEST_SPLIT),
                                   stratify=temp_df.diagnosis, random_state=SEED)
train_df, val_df, test_df = (d.reset_index(drop=True) for d in (train_df, val_df, test_df))
print(f"train {len(train_df)} | val {len(val_df)} | test {len(test_df)}")

class_weights = {int(c): float(w) for c, w in zip(
    range(NUM_CLASSES),
    compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=train_df.diagnosis))}
print("class weights:", {CLASS_NAMES[k]: round(v, 2) for k, v in class_weights.items()})

## 4. Input pipeline & fundus-safe augmentation

Flips and rotations are label-preserving for retinas (no canonical orientation); mild
brightness/contrast jitter mimics acquisition variability. We avoid shear/elastic warps that would
distort lesion morphology. Decoded images are cached in RAM, so only the first epoch pays decode cost.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.08, fill_mode="constant"),
    tf.keras.layers.RandomZoom(0.10, fill_mode="constant"),
    tf.keras.layers.RandomBrightness(0.10, value_range=(0, 255)),
    tf.keras.layers.RandomContrast(0.10),
], name="fundus_augmentation")

def _decode(path, label):
    img = tf.image.resize(tf.io.decode_png(tf.io.read_file(path), channels=3), (IMG_SIZE, IMG_SIZE))
    return tf.cast(img, tf.float32), label

def make_ds(df, training):
    ds = (tf.data.Dataset.from_tensor_slices((df.path.values, df.diagnosis.values.astype("int32")))
          .map(_decode, num_parallel_calls=AUTOTUNE).cache())
    if training:
        ds = (ds.shuffle(len(df), seed=SEED).batch(BATCH_SIZE)
              .map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE))
    else:
        ds = ds.batch(BATCH_SIZE)
    return ds.prefetch(AUTOTUNE)

train_ds, val_ds, test_ds = make_ds(train_df, True), make_ds(val_df, False), make_ds(test_df, False)
val_labels = val_df.diagnosis.values.astype("int32")

# Preview an augmented batch
xb, yb = next(iter(train_ds))
fig, axes = plt.subplots(1, 6, figsize=(16, 3))
for i, a in enumerate(axes):
    a.imshow(np.clip(xb[i].numpy(), 0, 255).astype("uint8"))
    a.set_title(CLASS_NAMES[yb[i]]); a.axis("off")
plt.suptitle("Augmented training batch"); plt.tight_layout(); plt.show()

## 5. Model — EfficientNetB0 + custom head

The backbone is built as a single flat graph (via `input_tensor=`) so every conv layer stays
addressable by name — needed for selective fine-tuning and Grad-CAM.

In [ ]:
def build_model():
    inputs = tf.keras.Input((IMG_SIZE, IMG_SIZE, 3), name="input_image")
    backbone = tf.keras.applications.EfficientNetB0(
        include_top=False, weights="imagenet", input_tensor=inputs)
    backbone.trainable = False
    x = tf.keras.layers.GlobalAveragePooling2D(name="head_gap")(backbone.output)
    x = tf.keras.layers.Dropout(DROPOUT, name="head_dropout")(x)
    out = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax", name="predictions")(x)
    return tf.keras.Model(inputs, out, name="efficientnetb0_dr"), backbone

model, backbone = build_model()
model.summary(show_trainable=True, line_length=90)

## 6. Quadratic Weighted Kappa callback

QWK needs the whole validation set at once, so we compute it in a callback and expose it as
`val_qwk`. Placed **first** in the callback list, it lets `EarlyStopping`/checkpointing select the
model that maximises the competition metric.

In [ ]:
def qwk(y_true, y_pred):
    return cohen_kappa_score(y_true, y_pred, weights="quadratic", labels=list(range(NUM_CLASSES)))

class QWKCallback(tf.keras.callbacks.Callback):
    def __init__(self, ds, y_true): super().__init__(); self.ds, self.y_true = ds, y_true
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        logs["val_qwk"] = qwk(self.y_true, np.argmax(self.model.predict(self.ds, verbose=0), 1))
        print(f" — val_qwk: {logs['val_qwk']:.4f}")

def callbacks(patience):
    return [QWKCallback(val_ds, val_labels),
            tf.keras.callbacks.ReduceLROnPlateau("val_loss", factor=0.3, patience=3, min_lr=1e-7, verbose=1),
            tf.keras.callbacks.EarlyStopping(monitor="val_qwk", mode="max", patience=patience,
                                             restore_best_weights=True, verbose=1)]

## 7. Phase 1 — train the head on a frozen backbone

In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(HEAD_LR),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
hist1 = model.fit(train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS,
                  class_weight=class_weights, callbacks=callbacks(HEAD_PATIENCE))

## 8. Phase 2 — fine-tune the top layers

Unfreeze the top `FINETUNE_UNFREEZE` backbone layers at a low learning rate. BatchNorm layers stay
frozen so their ImageNet statistics are preserved — the standard recipe for stable fine-tuning on a
small, domain-shifted dataset.

In [ ]:
for layer in backbone.layers:
    layer.trainable = False
for layer in backbone.layers[-FINETUNE_UNFREEZE:]:
    layer.trainable = True
for layer in model.layers:                       # keep BN frozen
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
print("trainable layers:", sum(l.trainable for l in model.layers))

model.compile(optimizer=tf.keras.optimizers.Adam(FINETUNE_LR),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
hist2 = model.fit(train_ds, validation_data=val_ds, epochs=FINETUNE_EPOCHS,
                  class_weight=class_weights, callbacks=callbacks(FINETUNE_PATIENCE))

## 9. Evaluation on the held-out test set

In [ ]:
def predict_paths(paths):
    probs = []
    for i in range(0, len(paths), BATCH_SIZE):
        imgs = np.stack([cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB).astype("float32")
                         for p in paths[i:i+BATCH_SIZE]])
        probs.append(model.predict(imgs, verbose=0))
    return np.concatenate(probs)

y_true = test_df.diagnosis.values.astype(int)
y_prob = predict_paths(test_df.path.values)
y_pred = np.argmax(y_prob, 1)

print(f"Test accuracy : {accuracy_score(y_true, y_pred):.4f}")
print(f"Test QWK      : {qwk(y_true, y_pred):.4f}")
print()
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

In [ ]:
# Confusion matrix (counts + row-normalised)
cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
cmn = cm / cm.sum(1, keepdims=True).clip(min=1)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, mat, title, fmt in [(axes[0], cm, "Counts", "d"), (axes[1], cmn, "Row-normalised", ".2f")]:
    im = ax.imshow(mat, cmap="Blues", vmax=mat.max() if fmt=="d" else 1.0)
    ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right"); ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"Confusion Matrix — {title}")
    th = (mat.max() if fmt=="d" else 1.0) / 2
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, format(mat[i,j], fmt), ha="center", va="center",
                    color="white" if mat[i,j] > th else "black")
plt.tight_layout(); plt.show()

In [ ]:
# One-vs-rest ROC curves + AUCs
y_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
plt.figure(figsize=(7, 6)); aucs = {}
for i, name in enumerate(CLASS_NAMES):
    if y_bin[:, i].sum() == 0: continue
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_prob[:, i])
    aucs[name] = roc_auc_score(y_bin[:, i], y_prob[:, i])
    plt.plot(fpr, tpr, label=f"{name} (AUC={aucs[name]:.3f})")
plt.plot([0,1],[0,1],"k--",alpha=.4); plt.xlabel("FPR"); plt.ylabel("TPR")
plt.title("One-vs-Rest ROC Curves"); plt.legend(loc="lower right"); plt.tight_layout(); plt.show()
print("Macro ROC-AUC:", round(float(np.mean(list(aucs.values()))), 4))

In [ ]:
# Training history
def cat(h1, h2, k): return h1.history.get(k, []) + h2.history.get(k, [])
b = len(hist1.history["loss"])
fig, ax = plt.subplots(1, 3, figsize=(18, 5))
for a, (tk, vk, t) in zip(ax, [("loss","val_loss","Loss"), ("accuracy","val_accuracy","Accuracy"),
                               (None,"val_qwk","Validation QWK")]):
    if tk: a.plot(cat(hist1,hist2,tk), label="train")
    a.plot(cat(hist1,hist2,vk), label="val"); a.axvline(b-0.5, color="grey", ls=":", label="fine-tune start")
    a.set_title(t); a.set_xlabel("epoch"); a.legend()
plt.tight_layout(); plt.show()

## 10. Grad-CAM — what is the model looking at?

Grad-CAM weights the final conv feature maps by the gradient of the predicted class, revealing the
regions that drove the decision. On fundus images these should coincide with clinically meaningful
lesions — microaneurysms, haemorrhages and exudates.

In [ ]:
LAST_CONV = next(l.name for l in reversed(model.layers) if isinstance(l, tf.keras.layers.Conv2D))

def gradcam(img_array, class_index=None):
    gm = tf.keras.Model(model.input, [model.get_layer(LAST_CONV).output, model.output])
    with tf.GradientTape() as tape:
        conv, preds = gm(img_array)
        class_index = int(tf.argmax(preds[0])) if class_index is None else class_index
        score = preds[:, class_index]
    grads = tape.gradient(score, conv)
    heat = tf.squeeze(conv[0] @ tf.reduce_mean(grads, (0,1,2))[..., None])
    return (tf.maximum(heat, 0) / (tf.reduce_max(heat) + 1e-8)).numpy(), class_index

def overlay(img, heat, alpha=0.4):
    img = img.astype(np.uint8)
    heat = cv2.resize(heat, (img.shape[1], img.shape[0]))
    retina = img.sum(2) > 10                      # suppress the black border
    heat = heat * retina
    heat = heat / heat.max() if heat.max() > 0 else heat
    col = cv2.cvtColor(cv2.applyColorMap(np.uint8(255*heat), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
    out = np.uint8(col*alpha + img*(1-alpha))
    out[~retina] = img[~retina]
    return out

fig, axes = plt.subplots(2, NUM_CLASSES, figsize=(4*NUM_CLASSES, 8))
for c in range(NUM_CLASSES):
    sub = test_df[test_df.diagnosis == c]; picked = None
    for _, r in sub.iterrows():
        im = cv2.cvtColor(cv2.imread(r.path), cv2.COLOR_BGR2RGB)
        h, p = gradcam(im.astype("float32")[None])
        if p == c: picked = (im, h); break
    if picked is None and len(sub):
        r = sub.iloc[0]; im = cv2.cvtColor(cv2.imread(r.path), cv2.COLOR_BGR2RGB)
        h, _ = gradcam(im.astype("float32")[None], class_index=c); picked = (im, h)
    im, h = picked
    axes[0, c].imshow(im); axes[0, c].set_title(CLASS_NAMES[c]); axes[0, c].axis("off")
    axes[1, c].imshow(overlay(im, h)); axes[1, c].axis("off")
axes[0,0].set_ylabel("original"); axes[1,0].set_ylabel("Grad-CAM")
plt.suptitle("Grad-CAM: regions driving each severity prediction"); plt.tight_layout(); plt.show()

## Summary

- **EfficientNetB0** transfer learning with a two-phase schedule (frozen-head → fine-tune top layers).
- **Class imbalance** handled with inverse-frequency weights; **QWK** used for model selection.
- Reported **accuracy, per-class precision/recall, confusion matrix, ROC-AUC and QWK** on a held-out
  test set, plus **Grad-CAM** explanations.

See the project [README](https://github.com/) for the modular `src/` implementation and CLI, and the
full write-up of results.

*Research/education only — not a medical device.*